# 🎙️ 100% Free Malayalam AI Neural Lip-Sync (Wav2Lip + Google Colab T4 GPU)
### ⚡ Unlimited Video Duration (Supports 8-10 Minutes Audio with Zero Cutoff & Pure Black Background)

ഈ നോട്ട്ബുക്ക് ഉപയോഗിച്ച് നിങ്ങളുടെ 8 മുതൽ 10 മിനിറ്റ് വരെയുള്ള വലിയ മലയാളം MP3 ഓഡിയോയും, അവതാരകരുടെ വീഡിയോയും (`man.mp4`, `women.mov`, `aunty.mp4`) നൽകി **ഒരു സെക്കൻഡ് പോലും കട്ടാവാതെ മുഴുവൻ സമയവും യഥാർത്ഥ ലിപ്-സിങ്ക് (Neural Lip Sync)** സൗജന്യമായി നിർമ്മിക്കാം.

---
### 🚀 നിർദ്ദേശങ്ങൾ (Instructions):
1. മുകളിലെ മെനുവിൽ നിന്ന് **Runtime -> Change runtime type -> T4 GPU** സെലക്ട് ചെയ്തിട്ടുണ്ടെന്ന് ഉറപ്പുവരുത്തുക.
2. താഴെ നൽകിയിരിക്കുന്ന 4 സ്റ്റെപ്പുകളിലെയും **Play (▶️)** ബട്ടൺ ക്രമമായി ക്ലിക്ക് ചെയ്യുക.
3. വീഡിയോ എത്ര മിനിറ്റ് ആണെങ്കിലും ഓഡിയോയുടെ മുഴുവൻ ദൈർഘ്യത്തിനും (8-10 മിനിറ്റ്) ചുണ്ടുകൾ ഒറിജിനലായി സംസാരിക്കുന്ന വീഡിയോ ഡൗൺലോഡ് ചെയ്യാം.

In [ ]:
# [Step 1] Install Dependencies & Download Wav2Lip GAN Weights
import os
%cd /content
if not os.path.exists('/content/Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git
%cd /content/Wav2Lip

# 1. Fix modern Librosa 0.10+ keyword argument requirements in audio.py
!sed -i 's/librosa.filters.mel(hp.sample_rate, hp.n_fft/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft/g' /content/Wav2Lip/audio.py
!sed -i 's/librosa.core.load/librosa.load/g' /content/Wav2Lip/audio.py

# 2. Fix unquoted ffmpeg calls in inference.py & create required directories
!sed -i 's/ffmpeg -y -i {} -strict -2 {}/ffmpeg -y -i "{}" -strict -2 "{}"/g' /content/Wav2Lip/inference.py
!sed -i 's/ffmpeg -y -i {} -i {} -strict -2 -q:v 1 {}/ffmpeg -y -i "{}" -i "{}" -strict -2 -q:v 1 "{}"/g' /content/Wav2Lip/inference.py
!mkdir -p /content/Wav2Lip/temp /content/Wav2Lip/results /content/Wav2Lip/checkpoints /content/Wav2Lip/face_detection/detection/sfd

# 3. Download Wav2Lip GAN Model from HuggingFace (Direct 200 OK link, 435MB)
!wget -c -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth"

# 4. Download S3FD Face Detection Model (86MB)
!wget -c -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

print('✅ [Step 1 Complete] GPU Environment, Fixes & AI Models Ready!')

In [ ]:
# [Step 2] Upload Presenter Video & 8-10 Minute Malayalam MP3
from google.colab import files
import subprocess, json, os

%cd /content/Wav2Lip
!mkdir -p /content/Wav2Lip/temp /content/Wav2Lip/results

print("1️⃣ നിങ്ങളുടെ അവതാരകന്റെ വീഡിയോ (man.mp4 / women.mov / aunty.mp4) അപ്‌ലോഡ് ചെയ്യുക:")
video_uploaded = files.upload()
raw_video_path = list(video_uploaded.keys())[0]

print("2️⃣ വെബ് ആപ്പിൽ നിന്നും നിർമ്മിച്ച 8-10 മിനിറ്റ് മലയാളം MP3 ഫയൽ അപ്‌ലോഡ് ചെയ്യുക:")
audio_uploaded = files.upload()
raw_audio_path = list(audio_uploaded.keys())[0]

# Convert audio directly to clean 16kHz mono WAV (Prevents shell syntax and special char errors)
clean_audio_path = "/content/Wav2Lip/input_audio_clean.wav"
!ffmpeg -y -i "{raw_audio_path}" -ar 16000 -ac 1 "{clean_audio_path}"

# Check Audio Duration using ffprobe
cmd_probe = f'ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 "{clean_audio_path}"'
audio_duration = float(subprocess.check_output(cmd_probe, shell=True).decode().strip())
print(f'📊 Audio Duration: {audio_duration:.1f} seconds ({audio_duration/60:.1f} minutes)')

# Auto-loop video seamlessly to match the exact audio length (Crucial for 8-10 mins audio)
looped_video = "/content/Wav2Lip/looped_input_video.mp4"
print("🔄 Looping video seamlessly to match full audio duration...")
!ffmpeg -y -stream_loop -1 -i "{raw_video_path}" -t {audio_duration} -c:v libx264 -preset ultrafast -crf 18 -an "{looped_video}"

print('✅ [Step 2 Complete] Video & Audio prepared cleanly for Full-Length Lip Sync!')

In [ ]:
# [Step 3] Run AI Neural Lip-Sync Inference (No Time Limit, Full 8-10 Mins)
%cd /content/Wav2Lip
!mkdir -p /content/Wav2Lip/temp /content/Wav2Lip/results

print('🚀 Neural Lip Sync generating... This processes every single frame with GPU acceleration.')

# Detect audio: clean WAV or uploaded audio
target_audio = "/content/Wav2Lip/input_audio_clean.wav"
if not os.path.exists(target_audio):
    target_audio = raw_audio_path

!python inference.py \
  --checkpoint_path /content/Wav2Lip/checkpoints/wav2lip_gan.pth \
  --face "/content/Wav2Lip/looped_input_video.mp4" \
  --audio "{target_audio}" \
  --outfile /content/Wav2Lip/results/final_malayalam_lipsync.mp4 \
  --wav2lip_batch_size 128 \
  --resize_factor 1 \
  --pads 0 10 0 0

print('🎉 [Step 3 Complete] 100% Neural Lip-Synced Video Generated!')

In [ ]:
# [Step 4] Download Your Final Lip-Synced Video (Pure Black Background)
from google.colab import files
import os

final_path = '/content/Wav2Lip/results/final_malayalam_lipsync.mp4'
if os.path.exists(final_path):
    size_mb = os.path.getsize(final_path) / (1024 * 1024)
    print(f'⬇️ Downloading Final Lip-Sync Video ({size_mb:.1f} MB)...')
    files.download(final_path)
else:
    print('❌ Output file not found. Please review Step 3 output for errors.')